In [ ]:
%load_ext autoreload
%autoreload 2


from kontext.visualization import space_plot,space_plot_mask,get_figs,get_proportions,plot_region_distribution,heatmap_proportions
from utils.load_data import load_and_preprocess_adata
from kontext.kontext_class import Kontext
from kontext.utils import lr_progression_df,get_LR_pairs

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
plt.rcParams['figure.facecolor'] = '#fcf8f500'
plt.rcParams['axes.facecolor'] = '#fcf8f500'

from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

In [ ]:

data = {'dataset':'LYMPH','sample':'3d'}  # Info to load the dataset 
preprocessing = {'preprocessing':'sparse_hvg_LR'} # Preprocessing params 
embedding = {"sigma_e":1, "snn":20, 'alpha':0.5} # Embedding params 
spatial_domains = {'clustering':'gmm','seed':0,'n_domains':2} # Spatial domains params


adata = load_and_preprocess_adata(data,preprocessing_params=preprocessing)
ko = Kontext(**embedding,verbose=True) # Init Kontext object 
# ko.weights(adata)
ko.spatial_domains(adata,**spatial_domains) # Compute the spatial domains 


In [ ]:
print(f"Cell types in presence :{adata.obs['annotation'].unique().tolist()}")
print(f"Shape of the dataset : {adata.shape}")
print(f"Number of sections : {adata.obs['n_section'].nunique()}")
print(f"Sections : {list(adata.obs['n_section'].unique())}")
print(f"Spatial coordinates : \
\n x in [{adata.obsm['spatial'][:,0].min():.2f},{adata.obsm['spatial'][:,0].max():.2f}]\
\n y in [{adata.obsm['spatial'][:,1].min():.2f},{adata.obsm['spatial'][:,1].max():.2f}]\
\n z in [{adata.obsm['spatial'][:,2].min():.2f},{adata.obsm['spatial'][:,2].max():.2f}]")

In [ ]:
s2 = 'spatial_domains'

ko.niche(adata, cell_type_key=s2)
conds = [(adata.obs[s2]==0)&(adata.obs[f'niche_{s2}_1']==0), 
        (adata.obs[s2]==0)&(adata.obs[f'niche_{s2}_1']>0), 
        (adata.obs[s2]==1)&(adata.obs[f'niche_{s2}_0']==0), 
        (adata.obs[s2]==1)&(adata.obs[f'niche_{s2}_0']>0)
        ]
adata.obs[f'region_{s2}'] = np.select(conds, ['TME', 'boundary', 'tumor', 'boundary'])
adata.uns[f'region_{s2}_colors'] = [ "#FFAD41","#1BA99B","#B34A44"]
adata.uns['annotation_colors'] = [
    '#3B75AF',
    '#EF8636',
    '#C53A32',
    '#9E47F3',
    '#84584E',
    '#B7BD6D',
    '#58BBCC',
    '#B3C6E5',
    '#F5BE82',
    '#A8DD93',
    '#F19D99',
]
adata.uns[f'{s2}_colors'] = [ "#FFAD41","#B34A44"]
adata.obsm['spatial'] = adata.obsm['spatial']*(-1,-1,1)


In [ ]:
fig,ax = get_figs(1)
space_plot(adata,'annotation',ax=ax)

## Figure 4B 

In [ ]:
sections = adata.obs['n_section'].unique()
prev_ = {sections[i]: sections[i-1] if i>0 else sections[i] for i in range(len(sections))}
section = 34

fig,ax  = get_figs(1,5,5)
# ax = axes[0]
adata_ = adata[adata.obs['n_section'].isin([section])].copy() 
mask = space_plot_mask(adata_, 'n_section', section,color='annotation', ax=ax, snm=1, sm=30)
ax.set_title(f"Cell types",fontsize=15)
ax.set_xlabel("")
ax.set_ylabel(f"Section {section}",fontsize=15)
ax.legend(fontsize=15,frameon=False,bbox_to_anchor=[1,1])

In [ ]:
section = 34
adata_ = adata[adata.obs['n_section'].isin([section])].copy() 
fig,ax  = get_figs(1,5,5)
mask = space_plot_mask(adata_, 'n_section', section,color=f'{s2}', ax=ax, snm=1, sm=30)
ax.set_title(f"Kontext Regions",fontsize=15)
ax.set_xlabel("")
ax.set_ylabel("")
ax.get_legend().remove()

## Figure 4D

In [ ]:
section = 34
adata_ = adata[adata.obs['n_section'].isin([section])].copy() 
fig,ax  = get_figs(1,5,5)
mask = space_plot_mask(adata_, 'n_section', section,color=f'region_{s2}', ax=ax, snm=1, sm=30)
ax.set_title(f"Kontext Regions",fontsize=15)
ax.set_xlabel("")
ax.set_ylabel("")
ax.get_legend().remove()

## Supplementary figure S13 

In [ ]:



# fig.savefig('kontext_project/figures/lymph/lymph_space_plots.pdf',bbox_inches='tight')

fig,ax  = get_figs(1,10,5)
# ax =axes[2]
proportions = get_proportions(adata,'annotation',f'{s2}',normalize=False)
plot_region_distribution(proportions,colors = adata.uns[f'{s2}_colors'] ,ax=ax,fontsize=12,threshold=.1)
print(proportions)
ax.set_xticklabels(ax.get_xticklabels(), rotation='horizontal', fontsize=15) 
ax.set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=15) 
ax.set_title('Region proportions',fontsize=15)
# ax.set_ylabel('$n_{obs}$',fontsize=15)
ax.set_xlabel('Normalized proportions',fontsize=15)
# ax.get_legend().remove()
ax.legend(fontsize=15,bbox_to_anchor=(1.2,1))
fig.tight_layout()
plt.show()
# fig.savefig('kontext_project/figures/lymph/lymph_regions_and_proportions.pdf',bbox_inches='tight')

## Figure 4E

In [ ]:

fig,ax  = get_figs(1,10,5)
# ax =axes[2]
proportions = get_proportions(adata,'annotation',f'region_{s2}',normalize=False)
plot_region_distribution(proportions,colors = adata.uns[f'region_{s2}_colors'] ,ax=ax,fontsize=12,threshold=.1)
print(proportions)
ax.set_xticklabels(ax.get_xticklabels(), rotation='horizontal', fontsize=15) 
ax.set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=15) 
ax.set_title('Region proportions',fontsize=15)
# ax.set_ylabel('$n_{obs}$',fontsize=15)
ax.set_xlabel('Normalized proportions',fontsize=15)
# ax.get_legend().remove()
fig.tight_layout()
ax.legend(fontsize=15,bbox_to_anchor=(1.5,1))
plt.show()

In [ ]:
import pyvista as pv
from scipy.spatial import cKDTree

coords = adata.obsm['spatial']
region_codes = adata.obs[f'region_{s2}'].astype('category').cat.codes.values
tree = cKDTree(coords)
_, nn_idx = tree.query(coords, k=2) 
nn_idx = nn_idx[:, 1]

region_3Dplot = {'boundary':{'color':"#1BA99B",'opacity': 1,'point_size':1},
                #  'TME':{'color':"#FFAD41",'opacity':0.02,'point_size':6},
                 'tumor':{'color':"#B34A44",'opacity':1,'point_size':6},}

pl = pv.Plotter()
for region,v in region_3Dplot.items():

    boundary_mask = adata.obs[f'region_{s2}']==region
    boundary_coords = coords[boundary_mask]

    cloud_boundary = pv.PolyData(boundary_coords)
    pl.add_mesh(
        cloud_boundary,
        color=v['color'],
        render_points_as_spheres=True,
        point_size=6,
        opacity=v['opacity'],
    )

pl.add_axes()
pl.show()



## Figure 4C

In [ ]:
print(pl.camera_position)

pl.camera_position = [(-23314, -22001,10469),(-8396, -4083, 550),(0.0, 0.0, 1.0)]
pl.show()
pl.camera_position = [(-23314, -22001,10469),(-8396, -4083, 550),(0.0, 0.0, 1.0)]
pl.show()
# a=pl.screenshot('kontext_project/figures/lymph/3D_plot1.tif',transparent_background=True)


print(pl.camera_position)
pl.camera_position = [(-12053, -20052, 5000), (-10015, -4015, -1000),(0.0, 0.0, 1.0)] 
pl.show()
pl.camera_position = [(-12053, -20052, 5000), (-10015, -4015, -1000),(0.0, 0.0, 1.0)] 
pl.show()
# a=pl.screenshot('kontext_project/figures/lymph/3D_plot2.tif',transparent_background=True)

In [ ]:
pairs = get_LR_pairs(adata, organism='human', verbose=False)
genes = pairs.index.tolist()
mode = 'mean'
# lr_normalize(adata, pairs)

In [ ]:
from kontext.utils import lr_normalize,lr_moments,lr_availability,lr_moments,lr_score,lr_aggregate,lr_cell_score,lr_log
import seaborn as sns
groups, ts, tr = 'annotation', 'CAF', 'Tumor cell'
mode, samples_ = 'mean', ['TME', 'boundary', 'tumor']
# mode, times_ = 'mean', ['sham', 'cryo7D', 'cryo28D']  

genes = pairs.index.tolist()
mode = 'mean'

pairs = get_LR_pairs(adata, organism='human', verbose=False)
ko.cell_cell_communication(adata,pairs,mode='mean')



groups = 'annotation'
t1, t2 = 'CAF', 'Tumor cell'
for receiver,sender in (t1, t2),(t2,t1),(t1, t1), (t2, t2): 
    ko.init_sender_receiver_communication(adata,pairs,sender,receiver,groups)


## Figure 4F

In [ ]:

send_recep = [(t1, t2),(t2,t1),(t1, t1), (t2, t2) ]
top = 30

genes_of_interest = [
'ADM_ACKR3', 
'CXCL8_SDC1', 
'ANGPTL4_SDC4', 
'ANGPTL4_SDC1', 
'VEGFA_EGFR', 
'VEGFA_CD44', 
'VEGFA_ITGB1', 
'VEGFC_ITGB1',

'ICAM4_ITGB1',
'F13A1_ITGB1',
# 'PRG4_CD44',
'CCL5_SDC1',
'HGF_SDC1',
'PTN_SDC2',
'HGF_SDC2',

'C3_CD46',

'JAG1_NOTCH3', 
'DLL1_NOTCH3',

'LGALS9_MET',
'ADAM10_MET',
'ADAM17_MET',
'SEMA5A_MET',
'SEMA4D_MET',
'DCN_MET',
'HGF_MET',
'LRIG1_MET',
'TIMP3_MET',]

genes = pairs.index.tolist()
fig, axes = plt.subplots(ncols=4,nrows=1,figsize=(24,5))
Ss = {}
i = -1
for ax, (s_, r_) in enumerate(send_recep ):
    i+=1
    print(i//2,i%2)
    # ax = axes[i//2,i%2]
    ax = axes[i]
    r= {'col':groups,'value':r_}

    if f'{s_}{r_}' in Ss:
        S = Ss[f'{s_}{r_}']
    else:
        S = lr_score(adata, s_, r_, groups, mode,r=r,src='P')
        Ss[f'{s_}{r_}'] = S
    print(s_,r_,S.shape)

    gi = pd.Index(adata.uns['lr']['genes']) # genes 
    pos = adata.obs_names.get_indexer(S.index) # positions of the obs of interest 
    det = pd.Series(np.asarray((adata.obsm['X_lr'][pos][:, gi.get_indexer(pairs.receptor)] > 0).mean(0)).ravel(),index=pairs.index) # percentage of cells expressing the receptors
    print('det',det[det.index.isin(genes_of_interest)])
    keep = det[det > .15].index#.intersection(prev.columns[(prev > .05).any()])   # receptor + ligand support
    T = lr_aggregate(adata, S, s_, by=f'region_{s2}')['all'].reindex(samples_)[[g for g in genes if (g in keep and g in S.columns)]]# or 'MET' in g]]

    # For suppp 
    print(s_,r_)
    exp = (lr_log(T).max())
    T = T[exp[exp>0.1].index]
    Rt1 = (lr_log(T).loc['tumor'] - lr_log(T).loc['boundary'])
    Rt2 = (lr_log(T).loc['boundary']-lr_log(T).loc['tumor'])
    final_genes = Rt1.nlargest(top).index.union(Rt2.nlargest(top).index) 

    print(len(final_genes))
    T = T[[g for g in genes_of_interest if g in T.columns]]
    Rt1 = (lr_log(T).loc['tumor'] - lr_log(T).loc['boundary'])
    T = T[Rt1.nlargest(len(T.T)).index]
    T = T.T[['boundary','tumor']].T
    print(lr_log(T).T)
    cbar = i==3
    hm = sns.heatmap((lr_log(T) if mode == 'mean' else T).T, ax=ax, cmap='RdBu_r', center=0, vmin=-2,vmax=2, 
                annot=True, fmt='.2f', annot_kws={'fontsize':15},
                linewidths=.5,cbar=cbar, cbar_kws={'label': 'Sender-receiver score' if mode == 'mean' else 'score'}) 
    if cbar:
        cbar_ax = hm.collections[0].colorbar
        cbar_ax.set_label(cbar_ax.ax.get_ylabel(), fontsize=20)


    ax.set_title(f'{s_} → {r_}', fontsize=20) 
    ax.set_xlabel('') 
    ylabel = 'LR pairs' if i==0 else ""
    ax.set_ylabel(ylabel,fontsize=15) 
    ax.set_xticklabels(ax.get_xticklabels(), rotation='horizontal', fontsize=20) 
    ax.set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=15) 

    # sec = ax.secondary_xaxis('top')
    # sec.set_xticks(ax.get_xticks())
    # sec.set_xticklabels(ax.get_xticklabels())
    # # sec.set_xticklabels([f'{s} to {r}' for s,r in pair_list], fontsize=20)
    # sec.tick_params(length=0)


fig.tight_layout() 
plt.show() 
# fig.savefig('kontext_project/figures/lymph/lymph_2g_CAF_Tumor_communication.pdf')

## Supplementary figure S10 and S11

In [ ]:
# for sections_ in sections[:5],sections[5:10],:
sections_ = sections
nrows = 4
ncols= 5
figsize = ncols*4 ,nrows * 5
for color in ['annotation','spatial_domains']:
    fig,axes = plt.subplots(ncols=ncols,nrows =nrows,figsize = figsize)
    for i,section in enumerate(np.sort(sections_)[::-1]): 
        adata_ = adata[adata.obs[f'n_section']==section].copy()
        adata_.obs['spatial_domains'] = adata_.obs['spatial_domains'].astype('category')
        ax = axes[i//5,i%5]
        space_plot(adata_,color,ax=ax)
        ax.set_title(f'section {section}',fontsize=15)
        ax.set_ylabel('',fontsize=15)
        ax.set_xlabel('',fontsize=15)
        ax.get_legend().remove()

            # ax = axes[1,i]
            # space_plot(adata_,'spatial_domains',ax=ax)
            # ax.get_legend().remove()
            # ax.set_title('',fontsize=15)
            # ax.set_ylabel('',fontsize=15)
            # ax.set_xlabel(f'section {section}',fontsize=15)
    fig.tight_layout()
    axes[3,-1].set_axis_off()
# axes[1,-1].set_axis_off()
    plt.show()

In [ ]:
for gene in adata.var_names:
    if gene in colors:
        print(gene)

In [ ]:
# for sections_ in sections[:5],sections[5:10],:
sections_ = sections
colors = ['VEGFA', f'region_{s2}']
ncols= len(colors)
nrows = 1

for _,section in enumerate(np.sort(sections_)[::-1]): 
    figsize = ncols*5 , 1*5
    fig,axes = plt.subplots(ncols=ncols,nrows =nrows,figsize = figsize)
    for i,color in enumerate(colors):
        adata_ = adata[adata.obs[f'n_section']==section].copy()
        adata_.obs['spatial_domains'] = adata_.obs['spatial_domains'].astype('category')
        ax = axes[i]
        space_plot(adata_,color,ax=ax)
        ax.set_title(f'section {section}',fontsize=15)
        ax.set_ylabel('',fontsize=15)
        ax.set_xlabel('',fontsize=15)
        # ax.get_legend().remove()

            # ax = axes[1,i]
            # space_plot(adata_,'spatial_domains',ax=ax)
            # ax.get_legend().remove()
            # ax.set_title('',fontsize=15)
            # ax.set_ylabel('',fontsize=15)
            # ax.set_xlabel(f'section {section}',fontsize=15)
    fig.tight_layout()
    # axes[3,-1].set_axis_off()
# axes[1,-1].set_axis_off()
    plt.show()

In [ ]:
from scipy.spatial import cKDTree

region_col = f'region_{s2}'                      # 'region_spatial_domains'
coords     = adata.obsm['spatial']               # (n, 3)

bnd = coords[(adata.obs[region_col] == 'boundary').values]
adata.obs['dist_boundary'] = cKDTree(bnd).query(coords, k=1)[0]   # 0 for boundary cells
sign = np.where(adata.obs[region_col].values == 'tumor', -1, 1)
adata.obs['signed_dist_boundary'] = sign * adata.obs['dist_boundary'].values

import scipy.sparse as sps

g = 'VEGFA'
g = 'ADM'
x = adata[:, g].X
e = np.asarray(x.todense()).ravel() if sps.issparse(x) else np.asarray(x).ravel()

d = adata.obs['signed_dist_boundary'].values      # or 'dist_boundary'
bins = pd.cut(d, np.quantile(d, np.linspace(0, 1, 31)), duplicates='drop')
df = pd.DataFrame({'d': d, 'e': e, 'bin': bins}).groupby('bin', observed=True).agg(
        d=('d', 'mean'), m=('e', 'mean'), s=('e', 'sem'))

fig, ax = get_figs(1, 6, 4)
ax.plot(df['d'], df['m'], '-o', ms=3, color='#B34A44')
ax.fill_between(df['d'], df['m'] - df['s'], df['m'] + df['s'], alpha=.3, color='#B34A44')
ax.axvline(0, ls='--', c='grey', lw=.8)
ax.set_xlabel('Signed distance to boundary (µm)'); ax.set_ylabel(f'{g} expression')
plt.show()


In [ ]:
for sections_ in sections[:10],sections[10:]:
    nrows = 2
    ncols=10
    figsize = ncols*4 ,nrows * 5
    fig,axes = plt.subplots(ncols=ncols,nrows =nrows,figsize = figsize)
    for i,section in enumerate(np.sort(sections_)): 
        adata_ = adata[adata.obs[f'n_section']==section].copy()
        adata_.obs['spatial_domains'] = adata_.obs['spatial_domains'].astype('category')
        ax = axes[0,i]
        space_plot(adata_,'annotation',ax=ax)
        ax.set_title('',fontsize=15)
        ax.set_ylabel('',fontsize=15)
        ax.set_xlabel('',fontsize=15)
        ax.get_legend().remove()

        ax = axes[1,i]
        space_plot(adata_,'spatial_domains',ax=ax)
        ax.get_legend().remove()
        ax.set_title('',fontsize=15)
        ax.set_ylabel('',fontsize=15)
        ax.set_xlabel(f'section {section}',fontsize=15)
    fig.tight_layout()
axes[0,-1].set_axis_off()
axes[1,-1].set_axis_off()
plt.show()

In [ ]:

cell_types = adata.obs['annotation'].sort_values().unique().tolist()
colors = adata.uns['annotation_colors']
regions = 'boundary','tumor','TME'
domains = 0,1


regions_3Dplot_ct = {}
for r in regions :
    for t,c in zip(cell_types,colors):
        regions_3Dplot_ct[f'{t}_{r}'] = {'color':c,'opacity':1,'point_size':1,'t':t,'r':r}

domains_3Dplot_ct = {}

for r in domains:
    for t,c in zip(cell_types,colors):
        domains_3Dplot_ct[f'{t}_{r}'] = {'color':c,'opacity':1,'point_size':1,'t':t,'r':r}
display(domains_3Dplot_ct)

## Supplementary figure S14

In [ ]:
import pyvista as pv
from scipy.spatial import cKDTree



coords = adata.obsm['spatial']
region_codes = adata.obs[f'region_{s2}'].astype('category').cat.codes.values
tree = cKDTree(coords)
_, nn_idx = tree.query(coords, k=2) 
nn_idx = nn_idx[:, 1]


adata.obs[f'annotation_region'] = pd.Categorical(adata.obs['annotation'].astype(str) + '_' + adata.obs[f'region_{s2}'].astype(str))
adata.obs[f'annotation_domain'] = pd.Categorical(adata.obs['annotation'].astype(str) + '_' + adata.obs[f'{s2}'].astype(str))
# print(adata.obs[f'annotation_domain'])





pl = pv.Plotter()
for region,v in domains_3Dplot_ct.items():
    print(region,v)

    if v['t'] in ['CAF','CAM','Keratin Pearl',"Tumor cell","Plasma IgG"]:
        if v['r'] == 1:
            boundary_mask = adata.obs[f'annotation_domain']==region
            print(region,boundary_mask.sum())
            boundary_coords = coords[boundary_mask]

            cloud_boundary = pv.PolyData(boundary_coords)
            pl.add_mesh(
                cloud_boundary,
                color=v['color'],
                render_points_as_spheres=True,
                point_size=6,
                opacity=v['opacity'],
            )

pl.add_axes()
pl.show()


print(pl.camera_position)

pl.camera_position = [(-23314, -22001,-10469),(-8396, -4083, 550),(0.0, 0.0, -1.0)]
pl.show()
pl.camera_position = [(-23314, -22001,-10469),(-8396, -4083, 550),(0.0, 0.0, -1.0)]
pl.show()
# a=pl.screenshot('kontext_project/figures/lymph/3D_plot1.tif',transparent_background=True)


print(pl.camera_position)
pl.camera_position = [(-4053, -20052, -5000), (-10015, -4015, -1000),(0.0, 0.0, -1.0)] 
pl.show()
pl.camera_position = [(-4053, -20052, -5000), (-10015, -4015, -1000),(0.0, 0.0, -1.0)] 
pl.show()
# a=pl.screenshot('kontext_project/figures/lymph/3D_plot2.tif',transparent_background=True)




In [ ]:

pl = pv.Plotter()
for region,v in regions_3Dplot_ct.items():
    print(region,v)

    if v['t'] in ['CAF','CAM','Keratin Pearl',"Tumor cell","Plasma IgG"]:
        if v['r'] == 'tumor':
            boundary_mask = adata.obs[f'annotation_region']==region
            print(region,boundary_mask.sum())
            boundary_coords = coords[boundary_mask]

            cloud_boundary = pv.PolyData(boundary_coords)
            pl.add_mesh(
                cloud_boundary,
                color=v['color'],
                render_points_as_spheres=True,
                point_size=6,
                opacity=v['opacity'],
            )

pl.add_axes()
pl.show()

pl.camera_position = [(-23314, -22001,-10469),(-8396, -4083, 550),(0.0, 0.0, -1.0)]
pl.show()
pl.camera_position = [(-23314, -22001,-10469),(-8396, -4083, 550),(0.0, 0.0, -1.0)]
pl.show()
# a=pl.screenshot('kontext_project/figures/lymph/3D_plot1.tif',transparent_background=True)


print(pl.camera_position)
pl.camera_position = [(-4053, -20052, -5000), (-10015, -4015, -1000),(0.0, 0.0, -1.0)] 
pl.show()
pl.camera_position = [(-4053, -20052, -5000), (-10015, -4015, -1000),(0.0, 0.0, -1.0)] 
pl.show()
# a=pl.screenshot('kontext_project/figures/lymph/3D_plot2.tif',transparent_background=True)



## Supplementary figure S16

In [ ]:
def align_binary_columns(adata, col1, col2, diff_col='diff'):
    c1 = adata.obs[col1].astype(int)
    c2 = adata.obs[col2].astype(int)

    # Flip col2 only if it currently disagrees with col1 more than it agrees
    flip = (c1 == c2).mean() < 0.5
    c2_aligned = 1 - c2 if flip else c2

    adata.obs[col2] = pd.Categorical(c2_aligned)

    adata.obs[diff_col] = (c1 != c2_aligned).astype(int)


sections_diff = [2,4,18,19,23,33,]



ncols = 3
nrows=len(sections_diff)
# nrows=len(sections)
figsize = ncols*7 ,nrows * 4

fig,axes = plt.subplots(ncols=ncols,nrows =nrows,figsize = figsize)

spatial_domains_2D = []
for i,section in enumerate(np.sort(sections_diff)): 
# for i,section in enumerate(np.sort(sections)): 

    adata_ = adata[adata.obs[f'n_section']==section]
    adata_.obs['spatial_domains_3D'] = adata_.obs['spatial_domains'].astype('category')
    adata_.uns['spatial_domains_3D_colors'] = adata_.uns['spatial_domains_colors']
    ko_section = Kontext(**embedding,verbose=True)
    ko_section.spatial_domains(adata_,**spatial_domains)
    ko_section.weights(adata_)

    align_binary_columns(adata_,'spatial_domains_3D','spatial_domains')
    spatial_domains_2D += [adata_.obs['spatial_domains']]

    ax = axes[i,0]
    space_plot(adata_,'spatial_domains_3D',ax=ax)
    title = f'3D spatial_domains' if i == 0 else ''
    ax.set_title(title,fontsize=25)
    ax.set_xlabel('',fontsize=15)
    ax.set_ylabel(f'section {section}',fontsize=25)
    # ax.get_legend().remove()


    ax = axes[i,1]
    space_plot(adata_,'spatial_domains',ax=ax)
    title = f'2D spatial_domains' if i == 0 else ''
    ax.set_title(title,fontsize=25)
    ax.set_xlabel('',fontsize=25)
    ax.set_ylabel('',fontsize=25)
    # ax.set_ylabel(f'section {section}',fontsize=25)

    ax = axes[i,2]
    space_plot(adata_,'diff',ax=ax)
    title = f'Difference \n {int(adata_.obs['diff'].sum())/len(adata_):.2f}% different' if i == 0 else f'{int(adata_.obs['diff'].sum())/len(adata_):.2f}% different'
    ax.set_title(title,fontsize=25)
    ax.set_xlabel('',fontsize=15)
    ax.set_ylabel('',fontsize=15)

fig.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns

cell_type_col = 'annotation'   # <-- replace with your actual cell-type column name

records = []

for i, section in enumerate(np.sort(sections)):
    adata_ = adata[adata.obs[f'n_section'] == section].copy()
    adata_.obs['spatial_domains_3D'] = adata_.obs['spatial_domains'].astype('category')
    adata_.uns['spatial_domains_3D_colors'] = adata_.uns['spatial_domains_colors']
    ko_section = Kontext(**embedding, verbose=True)
    ko_section.spatial_domains(adata_, **spatial_domains)
    ko_section.weights(adata_)

    align_binary_columns(adata_, 'spatial_domains_3D', 'spatial_domains')
    spatial_domains_2D += [adata_.obs['spatial_domains']]

    for ct in adata_.obs[cell_type_col].unique():
        mask = adata_.obs[cell_type_col] == ct
        records.append({
            'cell_type': ct,
            'section': section,
            '3D': adata_.obs.loc[mask, 'spatial_domains_3D'].astype(int).mean(),
            '2D': adata_.obs.loc[mask, 'spatial_domains'].astype(int).mean(),
        })



## Supplementary figure S15 

In [ ]:
df = pd.DataFrame(records)
df_long = df.melt(
    id_vars=['cell_type', 'section'],
    value_vars=['3D', '2D'],
    var_name='spatial_domains',
    value_name='proportion_domain_1'
)
print(df_long)
df_long = df_long.loc[df_long['cell_type'].isin(['Plasma IgG','Fibroblast','CAF','Macrophage'])]
fig, ax = plt.subplots(figsize=(5, 5))
sns.boxplot(data=df_long, x='cell_type', y='proportion_domain_1', hue='spatial_domains', ax=ax,palette={'3D': "#e75fa3", '2D': "#201588"},)
ax.set_ylabel('Proportion in domain 1')
ax.set_xlabel('Cell type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
adata.obsm

In [ ]:
import pyvista as pv
from scipy.spatial import cKDTree




genes_of_interest = [
'ADM_ACKR3', 
'CXCL8_SDC1', 
'ANGPTL4_SDC4', 
'ANGPTL4_SDC1', 
'VEGFA_EGFR', 
'VEGFA_CD44', 
'VEGFA_ITGB1', 
'VEGFC_ITGB1',

'ICAM4_ITGB1',
'F13A1_ITGB1',
# 'PRG4_CD44',
'CCL5_SDC1',
'HGF_SDC1',
'PTN_SDC2',
'HGF_SDC2',

'C3_CD46',

'JAG1_NOTCH3', 
'DLL1_NOTCH3',

'LGALS9_MET',
'ADAM10_MET',
'ADAM17_MET',
'SEMA5A_MET',
'SEMA4D_MET',
'DCN_MET',
'HGF_MET',
'LRIG1_MET',
'TIMP3_MET',]


cell_types = adata.obs['annotation'].sort_values().unique().tolist()
colors = adata.uns['annotation_colors']
regions = 'boundary','tumor','TME'
domains = 0,1


cell_types_3Dplot_ct = {}
for r in regions :
    for t,c in zip(cell_types,colors):
        regions_3Dplot_ct[f'{t}_{r}'] = {'color':c,'opacity':1,'point_size':1,'t':t,'r':r}


display(domains_3Dplot_ct)
coords = adata.obsm['spatial']
region_codes = adata.obs[f'region_{s2}'].astype('category').cat.codes.values
tree = cKDTree(coords)
_, nn_idx = tree.query(coords, k=2) 
nn_idx = nn_idx[:, 1]


adata.obs[f'annotation_region'] = pd.Categorical(adata.obs['annotation'].astype(str) + '_' + adata.obs[f'region_{s2}'].astype(str))
adata.obs[f'annotation_domain'] = pd.Categorical(adata.obs['annotation'].astype(str) + '_' + adata.obs[f'{s2}'].astype(str))
# print(adata.obs[f'annotation_domain'])





pl = pv.Plotter()
for region,v in domains_3Dplot_ct.items():
    print(region,v)

    if v['t'] in ['CAF','CAM','Keratin Pearl',"Tumor cell","Plasma IgG"]:
        if v['r'] == 1:
            boundary_mask = adata.obs[f'annotation_domain']==region
            print(region,boundary_mask.sum())
            boundary_coords = coords[boundary_mask]

            cloud_boundary = pv.PolyData(boundary_coords)
            pl.add_mesh(
                cloud_boundary,
                color=v['color'],
                render_points_as_spheres=True,
                point_size=6,
                opacity=v['opacity'],
            )

pl.add_axes()
pl.show()


print(pl.camera_position)

pl.camera_position = [(-23314, -22001,-10469),(-8396, -4083, 550),(0.0, 0.0, -1.0)]
pl.show()
pl.camera_position = [(-23314, -22001,-10469),(-8396, -4083, 550),(0.0, 0.0, -1.0)]
pl.show()
# a=pl.screenshot('kontext_project/figures/lymph/3D_plot1.tif',transparent_background=True)


print(pl.camera_position)
pl.camera_position = [(-4053, -20052, -5000), (-10015, -4015, -1000),(0.0, 0.0, -1.0)] 
pl.show()
pl.camera_position = [(-4053, -20052, -5000), (-10015, -4015, -1000),(0.0, 0.0, -1.0)] 
pl.show()
# a=pl.screenshot('kontext_project/figures/lymph/3D_plot2.tif',transparent_background=True)



In [ ]:
def lr_3Dplot_ct(adata, S, pair, regions, region_col='region',
                 cmap='reds', point_size=6, clim=None, per_region_clim=False,
                 zero_color='#1BA99B', zero_point_size=3, zero_opacity=1):
    v = pd.Series(np.nan, index=adata.obs_names, dtype=float)
    v.loc[S.index] = S[pair].values
    pos = v.values > 0
    # pos = v.values > 0.001 * v.values.max()
    print(pos.sum())
    clim = clim or (np.nanmin(v.values[pos]), np.nanmax(v.values))   # scale on non-zeros only
    d = {}
    for r in regions:
        m = (adata.obs[region_col] == r).values & v.notna().values
        cmap_region = {'TME':'inferno','boundary':'Greens','tumor':'Reds'}[r]
        zero_color_region = {'TME': "#FFAD41",'boundary':"#1BA99B", 'tumor':"#B34A44"}[r]
        for grp, mg in (('zero', m & ~pos), ('pos', m & pos)):
            if mg.sum() == 0:
                continue
            s = v.values[mg]
            print(grp,len(s))
            d[f'{pair}_{r}_{grp}'] = {
                'coords': adata.obsm['spatial'][mg], 'scalars': s,
                'cmap': cmap_region,
                'clim': (0, s.max()) if (per_region_clim and grp == 'pos') else clim,
                'color': zero_color_region if grp == 'zero' else None,
                'point_size': zero_point_size if grp == 'zero' else point_size,
                'opacity': zero_opacity if grp == 'zero' else 1,
                'r': r, 'grp': grp, 'n': int(mg.sum())}
    return d

for sender,receiver in (['CAF','Tumor cell'],['Tumor cell','CAF']):
    groups   = 'annotation'

    ko.init_sender_receiver_communication(adata, pairs, sender, receiver, groups)
    S = lr_score(adata, sender, receiver, groups, mode='mean',
                r={'col': groups, 'value': receiver}, src='P')   # cells x LR pairs



    for pair in genes_of_interest:
        lr_3Dplot = lr_3Dplot_ct(adata, S, pair, regions, region_col=f'region_{s2}')
        display(lr_3Dplot)


        region_shown = None         # None -> all regions

        pl = pv.Plotter()
        for k, v in lr_3Dplot.items():
            if region_shown is not None and v['r'] != region_shown:
                continue
            cloud = pv.PolyData(v['coords'])
            cloud[pair] = v['scalars']
            pl.add_mesh(cloud, scalars=pair, cmap=v['cmap'], clim=v['clim'],
                        render_points_as_spheres=True, point_size=v['point_size'],
                        opacity=v['opacity'], scalar_bar_args={'title': f'{pair}\n{sender}→{receiver}'})
        pl.add_axes()
        pl.camera_position = [(23314, 22001, -10469), (8396, 4083, 550), (0.0, 0.0, -1.0)]
        pl.show()

In [ ]:

send_recep = [(ts, tr), (tr, tr), (tr, ts), (ts, ts), ]


genes = pairs.index.tolist()
# fig, axes = get_figs(len(send_recep), 6, 5)
fig, axes = plt.subplots(ncols=2,nrows=2,figsize=(10,100))
Ss = {}
top = 100
i = -1
for ax, (s_, r_) in enumerate(send_recep ):
    i+=1
    print(i//2,i%2)
    ax = axes[i//2,i%2]
    r= {'col':groups,'value':r_}

    # Sender cells are filtered based on their presence in the neighborhood of the receivers 
    if f'{s_}{r_}' in Ss:
        S = Ss[f'{s_}{r_}']
    else:
        S = lr_score(adata, s_, r_, groups, mode,r=r,src='P')
        Ss[f'{s_}{r_}'] = S
    print(s_,r_,S.shape)
    # print(S.shape)
    # T = lr_aggregate(adata_, S, s_, by='sample')['all'].reindex(samples_)[[g for g in genes if g in S.columns]]

    gi = pd.Index(adata.uns['lr']['genes']) # genes 
    pos = adata.obs_names.get_indexer(S.index) # positions of the obs of interest 
    det = pd.Series(np.asarray((adata.obsm['X_lr'][pos][:, gi.get_indexer(pairs.receptor)] > 0).mean(0)).ravel(),index=pairs.index) # percentage of cells expressing the receptors
    print('det',det[det.index.isin(genes_of_interest)])
    keep = det[det > .15].index#.intersection(prev.columns[(prev > .05).any()])   # receptor + ligand support
    T = lr_aggregate(adata, S, s_, by=f'region_{s2}')['all'].reindex(samples_)[[g for g in genes if (g in keep and g in S.columns)]]# or 'MET' in g]]



    print(s_,r_)
    exp = (lr_log(T).max())
    T = T[exp[exp>0.1].index]
    # Rt1 = (lr_log(T).loc['tumor'] - lr_log(T).loc['boundary'])
    # Rt2 = (lr_log(T).loc['boundary']-lr_log(T).loc['tumor'])
    # final_genes = Rt1.nlargest(top).index.union(Rt2.nlargest(top).index) 
    # T = T[final_genes]

    # print(len(final_genes))
    # T = T[[g for g in genes if g in T.columns]]
    Rt1 = (lr_log(T).loc['tumor'] - lr_log(T).loc['boundary'])
    T = T[Rt1.nlargest(len(T.T)).index]
    T = T.T[['boundary','tumor']].T


    # print(det[final_genes])
    print(lr_log(T).T)
    cbar = i==3
    hm = sns.heatmap((lr_log(T) if mode == 'mean' else T).T, ax=ax, cmap='RdBu_r', center=0, vmin=-2,vmax=2, 
                annot=True, fmt='.2f', annot_kws={'fontsize':15},
                linewidths=.5,cbar=cbar, cbar_kws={'label': 'Sender-receiver score' if mode == 'mean' else 'score'}) 
    if cbar:
        cbar_ax = hm.collections[0].colorbar
        cbar_ax.set_label(cbar_ax.ax.get_ylabel(), fontsize=20)


    ax.set_title(f'{s_} → {r_}', fontsize=20) 
    ax.set_xlabel('') 
    ylabel = 'LR pairs' if i==0 else ""
    ax.set_ylabel(ylabel,fontsize=15) 
    ax.set_xticklabels(ax.get_xticklabels(), rotation='horizontal', fontsize=20) 
    ax.set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=15) 
fig.tight_layout() 
plt.show() 
# fig.savefig('kontext_project/figures/lymph/lymph_2g_CAF_Tumor_communication.pdf')

In [ ]:
groups, ts, tr, mode, region = 'annotation', 'CAF', 'Tumor cell', 'mean', f'region_{s2}'
mode = 'mean'
pairs = get_LR_pairs(adata, organism='human', verbose=False)
lr_normalize(adata, pairs)
lr_moments(adata, 'X_lr') 
lr_availability(adata, None)
lr_moments(adata, 'A_all')
for t in (ts, tr): 
    lr_availability(adata, t, groups)
    lr_moments(adata, f'A_{t}', om=(adata.obs[f'm_{t}'] > 0).values)
S = lr_score(adata, ts, tr, groups, mode)
lr_cell_score(adata, 'all', mode=mode, col='LR_all')

In [ ]:
# from kontext.utils import me_correlation
ko.smooth_and_pca(adata, "wc", "C",add_layer=True)
ko.local_alignment(adata, layer_C="C")

In [ ]:

from kontext.visualization import plot_ccc_score_per_group
fig, axes = get_figs(2, 6, 5)

ax = plot_ccc_score_per_group(adata, f'region_{s2}', 'LR_all','n_section',36, ax=axes[0], insersion=(.10, .12))
ax.axhline(1, ls='--', lw=.8, c='grey')
ax.set_title('Communication scores', fontsize=15); ax.set_ylabel('Mean per domain', fontsize=15); ax.set_xlabel(''); ax.set_yticks([])
ax.set_xticklabels([f'{t._text}' for t in ax.get_xticklabels()], rotation='horizontal', fontsize=15)


ax = plot_ccc_score_per_group(adata, f'region_{s2}', 'local_alignment', 'n_section',36, ax=axes[1], insersion=(.10, .12))
ax.set_title('Local alignment scores', fontsize=15); ax.set_ylabel(''); ax.set_xlabel(''); ax.set_yticks([])
ax.set_xticklabels([f'{t._text}' for t in ax.get_xticklabels()], rotation='horizontal', fontsize=15)
fig.tight_layout() 
plt.show()
#

In [ ]:
adata.uns.get(f'{region_col}_colors')

In [ ]:
def pie_region(adata, value, region_col, col='annotation', ax=None, colors=None):
    p = adata.obs.loc[adata.obs[col] == value, region_col].value_counts()#.reindex(['TME','boundary','tumor'])
    print(value,p)
    ax = ax or get_figs(1, 5, 5)[1]
    ax.pie(p.values, labels=[f'{k}\n{v:,}' for k, v in p.items()], autopct='%1.0f%%',
           colors=colors or adata.uns.get(f'{region_col}_colors'), startangle=90)
    ax.set_title(f'{value}  (n={p.sum():,})', fontsize=13)

fig, axes = get_figs(2, 5, 5)
for ax, t in zip(axes, [ts, tr]): pie_region(adata, t, f'{s2}', groups, ax=ax)
fig, axes = get_figs(2, 5, 5)
for ax, t,colors in zip(axes, [ts, tr],[[ '#1BA99B', '#B34A44','#FFAD41',],[  '#B34A44','#1BA99B','#FFAD41',]]): pie_region(adata, t, f'region_{s2}', groups, ax=ax,colors=colors)

In [ ]:
import scipy.sparse as sps
_D = lambda M: np.asarray(M.todense() if sps.issparse(M) else M, dtype=np.float32)
def _pairs_idx(adata):
    g, p = pd.Index(adata.uns['lr']['genes']), adata.uns['lr']['pairs']
    return g.get_indexer(p['ligand']), g.get_indexer(p['receptor']), p.index

def lr_support(adata, sender='all', receiver=None, groups='main_cell_type'):
    r = np.ones(adata.n_obs, bool) if receiver is None else (adata.obs[groups] == receiver).values
    return r & (adata.obs[f'm_{sender}'].values > 0)

def _norm(adata, src, om, cols, mode,r=None):                     # slice columns BEFORE densifying
    if r is None:
        M = _D(adata.obsm[src][om][:, cols])
        s = adata.uns['lr'][f'mom_{src}']
    else:
        M = _D(adata.obsm[src][om][:, cols])
        M = M[(adata[om].obs[r['col']]==r['value']).values]
        s = adata.uns['lr'][f'mom_{src}_{r['col']}_{r['value']}']

    if mode == 'raw':
        return(M)
    if mode == 'mean': 
        # return M / s['mu'][cols]
        return M / s['den'][cols]
    Z = (M - s['mu'][cols]) / s['sd'][cols]
    return np.clip(Z, 0, None) if mode == 'zplus' else Z


def _lr_dec_w(adata, s_, r_, groups, by, mode, gk,r=None):      # neff-weighted, consistent with lr_aggregate
    om, (li, ri, nm) = lr_support(adata, s_, r_, groups), _pairs_idx(adata)
    R = _norm(adata, 'X_lr', om, ri, mode,r=r)
    L = _norm(adata, f'A_{s_}', om, li, mode,r=r)
    w = adata.obs.loc[adata.obs_names[om], f'neff_{s_}'].values
    g = adata.obs.loc[adata.obs_names[om], by].astype(str).values
    f = lambda M: pd.DataFrame(M * w[:, None], index=g, columns=nm).groupby(level=0).sum().div(
                  pd.Series(w, index=g).groupby(level=0).sum(), axis=0).reindex(samples_)[gk]
    Rb, Lb, Sb = f(R), f(L), f(R * L)
    return Rb, Lb, Sb / (Rb * Lb).replace(0, np.nan)

for j, (s_, r_) in enumerate([(ts, tr), (tr, ts)]):
    fig, axes = get_figs(3,6,40)
    gk = [g for g in genes if g in adata.uns['lr']['pairs'].index]
    r = {'col':groups,'value':r_}
    for i, (M, nme, cm) in enumerate(zip(_lr_dec_w(adata, s_, r_, groups, f'region_{s2}', mode, gk,r=r),
                                         ['receptor  log2 R̄', 'availability  log2 L̄', 'coupling  log2(1+ρ)'],
                                         ['Reds', 'Blues', 'RdBu_r'])):
        M = np.log2(M.clip(lower=1e-3))
        sns.heatmap(M.T, ax=axes[i], cmap=cm, center=0 if 'coupling' in nme else None, annot=True, fmt='.1f',
                    annot_kws={'fontsize': 7}, linewidths=.5, mask=M.T.isna())
        axes[i].set_title(f'{s_} → {r_} — {nme}', fontsize=11)
        axes[i].set_xlabel('')
        axes[i].set_ylabel('')
        


In [ ]:
from mpl_toolkits.mplot3d import Axes3D   # registers the 3D projection
for sections_ in sections[:10],sections[10:]:
    # nrows = 2
    # ncols=10
    # figsize = ncols*4 ,nrows * 5
    # fig,axes = plt.subplots(ncols=ncols,nrows =nrows,figsize = figsize)
    fig,ax = get_figs(1)
    ax = fig.add_subplot(111, projection='3d')
    # Set the viewing angle
    for i,section in enumerate(np.sort(sections_)): 
        adata_ = adata[adata.obs[f'n_section']==section].copy()
        adata_.obs['spatial_domains'] = adata_.obs['spatial_domains'].astype('category')
        # ax = axes[0,i]
        ax.plot(adata_.obsm['spatial'][:,0], adata_.obsm['spatial'][:,1], zs=adata_.obsm['spatial'][:,2], zdir='z')
        ax.view_init(elev=20, azim=45)
        # space_plot(adata_,'annotation',ax=ax)
        ax.set_title('',fontsize=15)
        ax.set_ylabel('',fontsize=15)
        ax.set_xlabel('',fontsize=15)
    fig.tight_layout()
plt.show()

In [ ]:
np.min(adata.obsm['spatial'][:,2]),np.max(adata.obsm['spatial'][:,2])

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

def space_plot_3D_perspective(adata, color='annotation', ax=None, s=20, alpha=1,
                              spatial_layer='spatial', elev=20, azim=45, **kwargs):
    import scanpy as sc
    show = False
    if ax is None:
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        show = True

    sc.pl.embedding(
        adata,
        basis=spatial_layer,
        color=color,
        ax=ax,
        projection='3d',
        components='1,2,3',
        s=s,
        alpha=alpha,
        show=show,
        **kwargs
    )
    ax.view_init(elev=elev, azim=azim)
    return ax


for sections_ in sections[:], sections[10:]:


    for i, section in enumerate(np.sort(sections_)):

        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')

        adata_ = adata[adata.obs[f'n_section'] == section].copy()
        adata_.obs['spatial_domains'] = adata_.obs['spatial_domains'].astype('category')

        space_plot_3D_perspective(
            adata_,
            color=f'spatial_domains',
            ax=ax,
            spatial_layer='spatial',
            elev=20,
            azim=45,
        )
        ax.get_legend().remove()
        ax.set_title('')
        ax.set_axis_off()
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_zlabel('z')

        ax.set_axis_off()          # hides axes lines, ticks, and labels
        ax.grid(False)             # removes the 3D grid

        # make the 3 background panes transparent
        for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
            axis.pane.fill = False
            axis.pane.set_edgecolor('none')
            axis.line.set_color('none')     # hide the axis spine line

        # transparent figure/axes background
        fig.patch.set_visible(False)
        ax.patch.set_visible(False)

        plt.show()

In [ ]:

cell_types = adata.obs['annotation'].sort_values().unique().tolist()
colors = adata.uns['annotation_colors']
regions = 'boundary','tumor','TME'
domains = 0,1


all_3Dplot_ct = {}

for t,c in zip(cell_types,colors):
        all_3Dplot_ct[f'{t}'] = {'color':c,'opacity':.005,'point_size':5,'t':t}

pl = pv.Plotter()
for region,v in all_3Dplot_ct.items():
            print(region,v)
            if region in ['Tumor cell','Plasma IgG']:
                    v['opacity'] = 1
    # if v['t'] in ['CAF','CAM','Keratin Pearl',"Tumor cell","Plasma IgG"]:
    #     if v['r'] == 'tumor':
            boundary_mask = adata.obs[f'annotation']==region
            boundary_coords = coords[boundary_mask]
            print(region,boundary_mask.sum())

            cloud_boundary = pv.PolyData(boundary_coords)
            pl.add_mesh(
                cloud_boundary,
                color=v['color'],
                render_points_as_spheres=True,
                point_size=v['point_size'],
                opacity=v['opacity'],
            )

pl.add_axes()
pl.show()

pl.camera_position = [(23314, 22001,-10469),(8396, 4083, 550),(0.0, 0.0, -1.0)]
pl.show()
pl.camera_position = [(23314, 22001,-10469),(8396, 4083, 550),(0.0, 0.0, -1.0)]
pl.show()
# a=pl.screenshot('kontext_project/figures/lymph/3D_plot1.tif',transparent_background=True)


print(pl.camera_position)
pl.camera_position = [(4053, 20052, -5000), (10015, 4015, -1000),(0.0, 0.0, -1.0)] 
pl.show()
pl.camera_position = [(4053, 20052, -5000), (10015, 4015, -1000),(0.0, 0.0, -1.0)] 
pl.show()
# a=pl.screenshot('kontext_project/figures/lymph/3D_plot2.tif',transparent_background=True)



In [ ]:
r = {'col':groups,'value':tr}
Rb, Lb, K = _lr_dec_w(adata, ts, tr, groups, f'region_{s2}', mode, gk,r=r)
lg = lambda M: np.log2(M.clip(lower=1e-3))
print('dynamic range (sd over time × pairs, log2):',
      {'R': lg(Rb).values.std().round(2), 'L': lg(Lb).values.std().round(2), 'coupling': lg(K).values.std().round(2)})
print('identity check (should be ~0):', float((lg(Rb) + lg(Lb) + lg(K) - lg(Rb * Lb * K)).abs().max().max()))
print('BOUNDARY vs TME shift carried by each term (median over pairs):',
      {k: float((lg(M).loc['boundary'] - lg(M).loc['TME']).median()) for k, M in [('R', Rb), ('L', Lb), ('coupling', K)]})
fig.tight_layout(); plt.show()


In [ ]:
for s_ in ts,tr:
    print(s_)
    print(adata.obs.groupby(f'region_{s2}')[f'neff_{s_}'].describe()[['mean','25%','50%']])
    print(adata.obs.loc[S.index].groupby(f'region_{s2}').apply(lambda d: (d[f'neff_{s_}'] >= 3).mean()).round(3))

In [ ]:
groups, ts, tr, mode, region = 'annotation', 'CAF', 'Tumor cell', 'mean', f'region_{s2}'

pairs = get_LR_pairs(adata, organism='human', verbose=False)
lr_normalize(adata, pairs)
for t in (ts, tr): lr_availability(adata, t, groups)
lr_moments(adata, 'X_lr') 
[lr_moments(adata, f'A_{t}', om=(adata.obs[f'm_{t}'] > 0).values) for t in (ts, tr)]
S = lr_score(adata, ts, tr, groups, mode)


for t1,t2 in [[ts,tr],[tr,ts]]:
    S = lr_score(adata, t1, t2, groups, mode)
    by = f'region_{s2}'
    g = adata.obs.loc[S.index, by].astype(str).values
    n = pd.Series(g).value_counts()

    # T = (S > 0).groupby(g).sum().div(n, axis=0).reindex(regions)
    T = (S > 0).groupby(g).mean()#.div(n, axis=0).reindex(regions)

    top = 20
    ntot = (g==region).sum()
    print(region,ntot)
    T =T[T.max().nlargest(top).index]
    print(T.T)
    fig,ax=get_figs(1)
    ax.set_title(f'r{t2} s{t1}')
    sns.heatmap(T.T, ax=ax, cmap='RdBu_r',vmin=0, 
                annot=True, fmt='.2f',
                linewidths=.5, cbar_kws={'label': 'log2 obs/exp' if mode == 'mean' else 'score'})

In [ ]:
top = 10
fig,axes=get_figs(2,6,4)
i=-1
for t1,t2 in [[ts,tr],[tr,ts]]:

    S = lr_score(adata, t1, t2, groups, mode)
    g = adata.obs.loc[S.index, by].astype(str).values
    rec = adata.uns['lr']['pairs'].loc[S.columns, 'receptor']          # column -> receptor map

    P = (S > 0).T.groupby(rec.values).max().T.groupby(g).mean()        # regions × receptors

    T =P[P.sum().nlargest(top).index]
    print(T.T)

    i+=1
    ax = axes[i]
    ax.set_title(f'From {t1} to {t2}')
    sns.heatmap(T.T, ax=ax, cmap='Reds',vmin=0, vmax=0.75,
                annot=True, fmt='.2f',
                linewidths=.5, cbar_kws={'label':'proportion of potential receptions'})
    ax.set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=15) 
    ax.set_xticklabels(ax.get_xticklabels(), rotation='horizontal', fontsize=15) 
fig.tight_layout()
# fig.savefig('kontext_project/figures/lymph/lymph_communication.pdf',bbox_inches='tight')

    
# M = lr_aggregate(adata, S, ts, by=by).T.groupby(rec.values).mean().T
# n_pairs = rec.value_counts()

# j = pd.Index(adata.uns['lr']['genes']).get_indexer(rec.unique())
# B = pd.DataFrame((_D(adata.obsm['X_lr'][S.index.map(adata.obs_names.get_loc).values][:, j]) > 0),
#                  columns=rec.unique()).groupby(g).mean()

In [ ]:
import scipy.sparse as sps

adata.obs[f'nsend_{ts}'] = np.asarray((sps.csr_matrix(adata.obsm['wc']) > 0).multiply((adata.obs[groups] == ts).values).astype(np.float32))#.sum(1).ravel()

d = adata.obs.loc[S.index]

print(d.groupby(by)[f'nsend_{ts}'].describe()[['mean','50%']])

strat = (S > 0).groupby([d[by].values, d[f'nsend_{ts}'].clip(1, 5).values]).mean()   # region × n_senders

print(strat[T.columns[:6]].round(3))

In [ ]:
# --- cell 2: regional CAF <-> tumour heatmaps -------------------------------



mode ='mean' 
regions = ['TME', 'boundary', 'tumor']
top =  40
# for stat in [
    # 'all','prev',
            #  'pos']:
fig, axes = get_figs(2, 6, 8)
for ax, (s_, r_) in zip(axes, [(ts, tr), (tr, ts)]):
    S = lr_score(adata, s_, r_, groups, mode)
    T = lr_aggregate(adata, S, s_, by=region)
    Tall = T['all'].reindex(regions)
    Tprev = T['prev'].reindex(regions)
    Tpos = T['pos'].reindex(regions)

    Tall_ = Tprev 
    print(Tall_)
    T = Tall_[Tall_.sum(0).nlargest(top).index]
    # T = lr_log(T)
    
    sns.heatmap(T.T, ax=ax, cmap='RdBu_r',# center=0, 
                annot=True, fmt='.2f',
                linewidths=.5, cbar_kws={'label': 'log2 obs/exp' if mode == 'mean' else 'score'})
    ax.set_title(f'{s_} → {r_}', fontsize=13); ax.set_xlabel(''); ax.set_ylabel('LR pair')

fig.tight_layout()
plt.show()



In [ ]:
P = lr_aggregate(adata, S, ts, by=region, stat='prev').reindex(regions)      # prevalence, the other half
display(P[T.columns].round(3))
sns.heatmap(P[:,T.max(0).nlargest(top).index])

In [ ]:
communication_dict = {                  # communication_dict = {
    'CAF to Tumor':{'receiver':'Tumor cell','sender':'CAF'},
    'Tumor to CAF':{'receiver':'CAF','sender':'Tumor cell'}
    }           


In [ ]:
region = f'region_{s2}'

import seaborn as sns 
regions = ['TME','boundary','tumor']
dfs = []
titles = []

for k,v in communication_dict.items():
    sender,receiver= v['sender'],v['receiver']
    df = lr_progression_df(adata, sender, receiver, regions, region=region,k=20,zscore=False,nonzero_only=False)

    print(sender,receiver)
    print(df)
    dfs += [df]
    title = f'{sender} to {receiver}'
    titles += [title]    



In [ ]:
for df,title in zip(dfs,titles):
    fig,ax = get_figs(1)
    heatmap_proportions(df,ax=ax,scaled=False,)
    cbar = ax.collections[0].colorbar 
    cbar.ax.set_position([1.1, 0.2, 0.03, 0.6]) 
    ax.set_title(title)
    fig.tight_layout()
    plt.show()



In [ ]:
print(adata.uns['lr'].keys())
adata.uns['lr']['mom_X_lr']

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sps, seaborn as sns, matplotlib.pyplot as plt
from scipy.stats import spearmanr

_D = lambda M: np.asarray(M.todense() if sps.issparse(M) else M, dtype=np.float32)

def _pairs_idx(adata):
    g, p = pd.Index(adata.uns['lr']['genes']), adata.uns['lr']['pairs']
    return g.get_indexer(p['ligand']), g.get_indexer(p['receptor']), p.index

# (ii) depth-normalise -> log1p -> keep LR genes. No L2 (kernels only).
def lr_normalize(adata, pairs, target_sum=1e4, key='X_lr'):
    keep = pd.Index(sorted(set(pairs.ligand) | set(pairs.receptor))).intersection(adata.var_names)
    C = sps.csr_matrix(adata.layers['counts'], dtype=np.float32)
    C = sps.diags(target_sum / np.maximum(np.asarray(C.sum(1)).ravel(), 1)) @ C
    adata.obsm[key] = C[:, adata.var_names.get_indexer(keep)].log1p().tocsr()
    adata.uns['lr'] = {'genes': list(keep), 'pairs': pairs[pairs.ligand.isin(keep) & pairs.receptor.isin(keep)]}

# (iii) availability A = W^(t)X, niche mass m, kappa^2, Kish neff. Row-normalise BEFORE masking.
def lr_availability(adata, sender, groups='main_cell_type', w_key='wc', key='X_lr'):
    W = sps.csr_matrix(adata.obsm[w_key], dtype=np.float32)
    W = sps.diags(1 / np.maximum(np.asarray(W.sum(1)).ravel(), 1e-12)) @ W
    t = (adata.obs[groups] == sender).values.astype(np.float32)
    m, k2 = W @ t, W.multiply(W) @ t
    adata.obs[f'm_{sender}'], adata.obs[f'neff_{sender}'] = m, np.where(k2 > 0, m**2 / np.maximum(k2, 1e-24), 0.)
    
    adata.obsm[f'A_{sender}'] = (W @ adata.obsm[key].multiply(t[:, None])).tocsr()

# (iv)+(v) pooled per-gene moments of an obsm block, over the support om (default: all cells).
def lr_moments(adata, src, om=None, q=0.05):
    M = adata.obsm[src] if om is None else adata.obsm[src][om]
    mu = np.asarray(M.mean(0)).ravel().astype(np.float32)
    ex2 = np.asarray(M.multiply(M).mean(0)).ravel() if sps.issparse(M) else (_D(M)**2).mean(0)
    sd = np.sqrt(np.maximum(ex2 - mu**2, 0)).astype(np.float32)
    adata.uns['lr'][f'mom_{src}'] = {'mu': mu, 'sd': sd + 1e-9, 'den': mu + np.quantile(mu[mu > 0], q)}

def lr_support(adata, sender, receiver, groups='main_cell_type'):
    return ((adata.obs[groups] == receiver) & (adata.obs[f'm_{sender}'] > 0)).values

def _norm(adata, src, om, cols, mode):
    M, s = _D(adata.obsm[src][om])[:, cols], adata.uns['lr'][f'mom_{src}']
    if mode == 'mean': return M / s['den'][cols]
    Z = (M - s['mu'][cols]) / s['sd'][cols]
    return np.clip(Z, 0, None) if mode == 'zplus' else Z

# (vi)=(viii): S_i = f(X_iR) * f(A_iL) on Omega = {receiver cells with >=1 sender neighbour}
def lr_score(adata, sender, receiver, groups='main_cell_type', mode='mean'):
    om, (li, ri, names) = lr_support(adata, sender, receiver, groups), _pairs_idx(adata)
    R, L = _norm(adata, 'X_lr', om, ri, mode), _norm(adata, f'A_{sender}', om, li, mode)
    return pd.DataFrame(R * L, index=adata.obs_names[om], columns=names)

# (vii) neff-weighted aggregation; stat in {'all','prev','pos'} with all = prev * pos
def lr_aggregate(adata, S, sender, by=None, stat='all'):
    w = adata.obs.loc[S.index, f'neff_{sender}'].values
    g = np.array(['all'] * len(S)) if by is None else adata.obs.loc[S.index, by].astype(str).values
    num, den = S.mul(w, axis=0).groupby(g).sum(), pd.Series(w, index=S.index).groupby(g).sum()
    pos = (S > 0).mul(w, axis=0).groupby(g).sum()
    return {'all': num.div(den, axis=0), 'prev': pos.div(den, axis=0), 'pos': num / pos.replace(0, np.nan)}[stat]

# (ix)
def lr_log(T, floor=1e-2): return np.log2(T.clip(lower=floor))

In [ ]:
def dx_qc(adata):
    C = adata.layers['counts']
    adata.obs['total_counts'], adata.obs['n_genes'] = np.asarray(C.sum(1)).ravel(), np.asarray((C > 0).sum(1)).ravel()

def dx_depth(adata, S, cols=('total_counts', 'n_genes')):                     # 1. depth confounder
    s = S.mean(1); return pd.Series({c: spearmanr(s, adata.obs.loc[S.index, c])[0] for c in cols})

def dx_neff(adata, S, sender):                                                # 2. neighbourhood-size confounder
    return spearmanr(S.mean(1), adata.obs.loc[S.index, f'neff_{sender}'])[0]

def dx_scramble(adata, sender, receiver, groups='main_cell_type', mode='mean', seed=0):   # 3. scrambled pairs
    p = adata.uns['lr']['pairs']
    adata.uns['lr']['pairs'] = p.assign(receptor=np.random.default_rng(seed).permutation(p.receptor.values))
    S = lr_score(adata, sender, receiver, groups, mode); adata.uns['lr']['pairs'] = p
    return S

def dx_downsample(adata, pairs, sender, receiver, groups='main_cell_type', frac=.5, seed=0, mode='mean'):  # 4.
    a = adata.copy(); C = sps.csr_matrix(a.layers['counts'])
    C.data = np.random.default_rng(seed).binomial(C.data.astype(int), frac).astype(np.float32)
    a.layers['counts'] = C; lr_normalize(a, pairs); lr_availability(a, sender, groups)
    lr_moments(a, 'X_lr'); lr_moments(a, f'A_{sender}', om=(a.obs[f'm_{sender}'] > 0).values)
    return lr_score(a, sender, receiver, groups, mode)

def dx_sections(adata, S, sender, by='sample'):                               # 5. replicate concordance
    return lr_aggregate(adata, S, sender, by=by).T.corr(method='spearman')

def dx_redundancy(adata, S, top=20):                                          # 6. shared-partner redundancy
    p = adata.uns['lr']['pairs'].loc[S.mean().nlargest(top).index]
    return p.groupby('receptor')['ligand'].apply('|'.join)

def dx_quadrants(adata, sender, receiver, groups='main_cell_type'):           # 7. double-negative occupancy
    om, (li, ri, _) = lr_support(adata, sender, receiver, groups), _pairs_idx(adata)
    R, L = _norm(adata, 'X_lr', om, ri, 'z') > 0, _norm(adata, f'A_{sender}', om, li, 'z') > 0
    return pd.Series({'++': (R & L).mean(), '+-': (R & ~L).mean(), '-+': (~R & L).mean(), '--': (~R & ~L).mean()})

# (x) wrapper
def lr_diagnostics(adata, sender, receiver, groups='main_cell_type', mode='mean'):
    S, F = lr_score(adata, sender, receiver, groups, mode), dx_scramble(adata, sender, receiver, groups, mode)
    return S, pd.Series({**dx_depth(adata, S), 'neff': dx_neff(adata, S, sender),
                         'real/scrambled': S.values.mean() / F.values.mean(),
                         'frac_Omega': len(S) / max((adata.obs[groups] == receiver).sum(), 1)})

In [ ]:
# --- cell 1: run ------------------------------------------------------------


In [ ]:
S = lr_score(adata, ts, tr, groups, mode)                                    # recompute: S/T above are the last direction
P = lr_aggregate(adata, S, ts, by=region, stat='prev').reindex(regions)[T.columns]
fig, ax = get_figs(1, 6, 8)
sns.heatmap(P.T, ax=ax, cmap='viridis', vmin=0, vmax=1, annot=True, fmt='.2f', linewidths=.5,
            cbar_kws={'label': 'prevalence (frac. cells S>0)'})
ax.set_title(f'{ts} → {tr} — prevalence', fontsize=13); ax.set_xlabel(''); ax.set_ylabel('LR pair')
fig.tight_layout(); plt.show()

In [ ]:
mode, regions, top = 'mean', ['TME', 'boundary', 'tumor'], 20

fig, axes = plt.subplots(2, 2, figsize=(12, 16))
for j, (s_, r_) in enumerate([(ts, tr), (tr, ts)]):
    S = lr_score(adata, s_, r_, groups, mode)
    T = lr_aggregate(adata, S, s_, by=region).reindex(regions); T = T[T.max(0).nlargest(top).index]
    P = lr_aggregate(adata, S, s_, by=region, stat='prev').reindex(regions)[T.columns]
    sns.heatmap((lr_log(T) if mode == 'mean' else T).T, ax=axes[0, j], cmap='RdBu_r', center=0, annot=True,
                fmt='.2f', linewidths=.5, cbar_kws={'label': 'log2 obs/exp' if mode == 'mean' else 'score'})
    sns.heatmap(P.T, ax=axes[0 + 1, j], cmap='viridis', annot=True, fmt='.6f',
                linewidths=.5, cbar_kws={'label': 'prevalence (frac. cells S>0)'})
    for i, lab in enumerate(['intensity', 'prevalence']):
        axes[i, j].set_title(f'{s_} → {r_} — {lab}', fontsize=13); axes[i, j].set_xlabel(''); axes[i, j].set_ylabel('LR pair')
fig.tight_layout(); plt.show()